# Language Identification for IndicCorpV2 — Built From Scratch

This single notebook does everything end-to-end:

1. Downloads sentences from **ai4bharat/IndicCorpV2** (23 language/script configs, the repo's own labels)
2. Splits paragraphs into sentences, samples 1000 sentences/language, stratified 80/10/10 split
3. Builds **TF-IDF vectors from scratch** (word 1-2 grams + char 2/3/4-grams) — no `sklearn.feature_extraction`
4. Trains a **multiclass logistic regression from scratch** (softmax + gradient descent) — no `sklearn.linear_model`
5. Evaluates with **macro-F1 computed from scratch** — no `sklearn.metrics`

**Before running:** in Kaggle, open the notebook Settings (right sidebar) and turn **Internet ON** — step 1 needs to reach the Hugging Face Hub.

You can tune the constants in the **Config** cell below (number of languages, sentences/language, epochs, etc.) before running everything.

## 0. Install & imports

In [31]:
!pip install -q datasets huggingface_hub


In [32]:
import os
import re
import json
import math
import random
import pickle

import numpy as np
from scipy import sparse
from collections import Counter

from datasets import load_dataset, get_dataset_config_names, get_dataset_split_names


## 1. Config

Adjust these before running the full pipeline.

In [33]:
REPO = "ai4bharat/IndicCorpV2"

N_LANGS = 23          # number of language/script configs to use
N_PER_LANG = 1000     # sentences to sample per language
OUT_DIR = "./langid_data"
SEED = 42

# TF-IDF
MIN_DF = 2
MAX_FEATURES = 200_000

# Logistic regression
LR = 0.5
L2 = 1e-4
EPOCHS = 30
BATCH_SIZE = 256

random.seed(SEED)
np.random.seed(SEED)
os.makedirs(OUT_DIR, exist_ok=True)


## 2. Data preparation

Pulls IndicCorpV2's own configs (each one *is* the official language+script label, e.g. `hin_Deva`, `kas_Arab`, `mni_Beng`), streams documents, splits paragraphs into sentences (Indic sentence terminators `।` `॥` in addition to `.!?`, plus newlines), samples `N_PER_LANG` sentences per language, then performs a per-label stratified 80/10/10 split.

In [34]:
_WORD_RE = re.compile(r"\S+", re.UNICODE)
_SENT_SPLIT_RE = re.compile(r"(?<=[।॥.!?])\s+|\n+")


def split_into_sentences(paragraph: str):
    paragraph = paragraph.strip()
    if not paragraph:
        return []
    parts = _SENT_SPLIT_RE.split(paragraph)
    sents = [p.strip() for p in parts if p and len(p.strip()) > 0]
    sents = [s for s in sents if len(s.split()) >= 3]
    return sents


def collect_sentences_for_config(split_name, n_needed, seed, max_docs_scan=20000):
    rng = random.Random(seed)
    ds = load_dataset(REPO, CONFIG_NAME, split=split_name, streaming=True)

    pool = []
    docs_scanned = 0
    for row in ds:
        text = row.get("text", "")
        docs_scanned += 1
        sents = split_into_sentences(text)
        pool.extend(sents)
        if len(pool) >= n_needed * 5 or docs_scanned >= max_docs_scan:
            break

    if len(pool) < n_needed:
        print(f"  [warn] only found {len(pool)} sentences for {split_name} (needed {n_needed}); using all of them.")
        rng.shuffle(pool)
        return pool

    rng.shuffle(pool)
    return pool[:n_needed]


def stratified_split(sentences, labels, train_frac=0.8, val_frac=0.1, seed=42):
    rng = random.Random(seed)
    by_label = {}
    for s, l in zip(sentences, labels):
        by_label.setdefault(l, []).append(s)

    train, val, test = [], [], []
    for lab, sents in by_label.items():
        sents = sents[:]
        rng.shuffle(sents)
        n = len(sents)
        n_train = int(round(n * train_frac))
        n_val = int(round(n * val_frac))
        train_s = sents[:n_train]
        val_s = sents[n_train:n_train + n_val]
        test_s = sents[n_train + n_val:]
        train += [(s, lab) for s in train_s]
        val += [(s, lab) for s in val_s]
        test += [(s, lab) for s in test_s]

    rng.shuffle(train)
    rng.shuffle(val)
    rng.shuffle(test)
    return train, val, test


Run the collection. This step needs internet and can take a while (streaming 23 languages).

In [35]:
all_configs = get_dataset_config_names(REPO)
print(f"Available configs in {REPO}: {all_configs}")

CONFIG_NAME = all_configs[0]
all_langs = sorted(get_dataset_split_names(REPO, CONFIG_NAME))
langs = all_langs[:N_LANGS]
print(f"Using {len(langs)} language splits (of {len(all_langs)} available) from config '{CONFIG_NAME}':")
print(langs)

all_sents, all_labels = [], []
for i, split_name in enumerate(langs):
    print(f"[{i+1}/{len(langs)}] collecting sentences for '{split_name}' ...")
    sents = collect_sentences_for_config(split_name, N_PER_LANG, seed=SEED + i)
    all_sents += sents
    all_labels += [split_name] * len(sents)
    print(f"  -> got {len(sents)} sentences")

print(f"\nTotal sentences collected: {len(all_sents)}")

Available configs in ai4bharat/IndicCorpV2: ['indiccorp_v2']
Using 23 language splits (of 23 available) from config 'indiccorp_v2':
['asm_Beng', 'ben_Beng', 'brx_Deva', 'doi_Deva', 'gom_Deva', 'guj_Gujr', 'hin_Deva', 'kan_Knda', 'kas_Arab', 'khasi', 'mai_Deva', 'mal_Mlym', 'mar_Deva', 'mni_Mtei', 'npi_Deva', 'ory_Orya', 'pan_Guru', 'san_Deva', 'santhali', 'snd_Deva', 'tam_Taml', 'tel_Telu', 'urd_Arab']
[1/23] collecting sentences for 'asm_Beng' ...
  -> got 1000 sentences
[2/23] collecting sentences for 'ben_Beng' ...
  -> got 1000 sentences
[3/23] collecting sentences for 'brx_Deva' ...
  -> got 1000 sentences
[4/23] collecting sentences for 'doi_Deva' ...
  -> got 1000 sentences
[5/23] collecting sentences for 'gom_Deva' ...
  -> got 1000 sentences
[6/23] collecting sentences for 'guj_Gujr' ...
  -> got 1000 sentences
[7/23] collecting sentences for 'hin_Deva' ...
  -> got 1000 sentences
[8/23] collecting sentences for 'kan_Knda' ...
  -> got 1000 sentences
[9/23] collecting sentence

In [36]:
train, val, test = stratified_split(all_sents, all_labels, train_frac=0.8, val_frac=0.1, seed=SEED)

for name, split in (("train", train), ("val", val), ("test", test)):
    path = os.path.join(OUT_DIR, f"{name}.jsonl")
    with open(path, "w", encoding="utf-8") as f:
        for text, label in split:
            f.write(json.dumps({"text": text, "label": label}, ensure_ascii=False) + "\n")
    print(f"wrote {len(split)} rows -> {path}")

train_texts, train_labels = [t for t, l in train], [l for t, l in train]
val_texts, val_labels = [t for t, l in val], [l for t, l in val]
test_texts, test_labels = [t for t, l in test], [l for t, l in test]


wrote 18400 rows -> ./langid_data/train.jsonl
wrote 2300 rows -> ./langid_data/val.jsonl
wrote 2300 rows -> ./langid_data/test.jsonl


## 3. Custom TF-IDF vectorizer (from scratch)

Features combined into one vector per sentence:
- word unigrams + word bigrams (whitespace tokenization)
- character 2/3/4-grams (computed per word with `^`/`$` boundary markers so grams never cross word boundaries)

Weighting: sublinear TF (`1 + log(tf)`), smoothed IDF `ln((1+N)/(1+df)) + 1`, L2-normalized rows.
Stored as `scipy.sparse` matrices (a data structure, not a modeling/vectorizing class) for efficiency.
No `sklearn.feature_extraction.*` is used.

In [37]:
def word_tokenize(text):
    return _WORD_RE.findall(text)


def char_ngrams(text, n):
    grams = []
    for w in word_tokenize(text):
        padded = f"^{w}$"
        if len(padded) < n:
            grams.append(padded)
            continue
        for i in range(len(padded) - n + 1):
            grams.append(padded[i:i + n])
    return grams


def word_ngrams(tokens, n):
    if len(tokens) < n:
        return []
    return [" ".join(tokens[i:i + n]) for i in range(len(tokens) - n + 1)]


def extract_features(text):
    tokens = word_tokenize(text)
    feats = Counter()
    for tok in word_ngrams(tokens, 1):
        feats[("W1", tok)] += 1
    for tok in word_ngrams(tokens, 2):
        feats[("W2", tok)] += 1
    for n in (2, 3, 4):
        for g in char_ngrams(text, n):
            feats[(f"C{n}", g)] += 1
    return feats


class CustomTfidfVectorizer:
    def __init__(self, min_df=2, max_features=200_000, sublinear_tf=True):
        self.min_df = min_df
        self.max_features = max_features
        self.sublinear_tf = sublinear_tf
        self.vocabulary_ = None
        self.idf_ = None

    def _doc_features(self, documents):
        return [extract_features(doc) for doc in documents]

    def fit(self, documents):
        self.fit_transform(documents)
        return self

    def fit_transform(self, documents):
        doc_feats = self._doc_features(documents)

        df = Counter()
        for feats in doc_feats:
            for f in feats.keys():
                df[f] += 1

        df = {f: c for f, c in df.items() if c >= self.min_df}

        if self.max_features is not None and len(df) > self.max_features:
            top = sorted(df.items(), key=lambda kv: (-kv[1], kv[0]))[: self.max_features]
            df = dict(top)

        vocab = {feat: idx for idx, feat in enumerate(sorted(df.keys()))}
        self.vocabulary_ = vocab

        n_docs = len(documents)
        n_feats = len(vocab)
        idf = np.zeros(n_feats, dtype=np.float64)
        for feat, idx in vocab.items():
            idf[idx] = math.log((1 + n_docs) / (1 + df[feat])) + 1.0
        self.idf_ = idf

        return self._build_matrix(doc_feats)

    def transform(self, documents):
        if self.vocabulary_ is None:
            raise RuntimeError("Vectorizer must be fit before calling transform().")
        doc_feats = self._doc_features(documents)
        return self._build_matrix(doc_feats)

    def _build_matrix(self, doc_feats):
        vocab = self.vocabulary_
        rows, cols, data = [], [], []
        for r, feats in enumerate(doc_feats):
            for feat, tf in feats.items():
                idx = vocab.get(feat)
                if idx is None:
                    continue
                tf_w = (1.0 + math.log(tf)) if self.sublinear_tf else float(tf)
                rows.append(r)
                cols.append(idx)
                data.append(tf_w)

        n_rows = len(doc_feats)
        n_cols = len(vocab)
        X = sparse.csr_matrix((data, (rows, cols)), shape=(n_rows, n_cols), dtype=np.float64)

        X = X.multiply(self.idf_)
        X = sparse.csr_matrix(X)

        norms = np.sqrt(X.multiply(X).sum(axis=1)).A1
        norms[norms == 0] = 1.0
        inv_norms = sparse.diags(1.0 / norms)
        X = inv_norms.dot(X)
        return sparse.csr_matrix(X)


## 4. Custom logistic regression (from scratch)

Multiclass softmax regression: `softmax(XW + b)`, trained by mini-batch gradient descent on cross-entropy loss with L2 weight decay and per-epoch learning-rate decay. Pure numpy; no `sklearn.linear_model.*` is used.

In [38]:
def softmax(Z):
    Z = Z - Z.max(axis=1, keepdims=True)
    expZ = np.exp(Z)
    return expZ / expZ.sum(axis=1, keepdims=True)


class CustomLogisticRegression:
    def __init__(self, lr=0.5, l2=1e-4, n_epochs=30, batch_size=256,
                 lr_decay=0.98, random_state=42, verbose=True):
        self.lr = lr
        self.l2 = l2
        self.n_epochs = n_epochs
        self.batch_size = batch_size
        self.lr_decay = lr_decay
        self.random_state = random_state
        self.verbose = verbose
        self.W = None
        self.b = None
        self.classes_ = None
        self.class_to_idx_ = None

    def _one_hot(self, y_idx, n_classes):
        n = len(y_idx)
        Y = np.zeros((n, n_classes), dtype=np.float64)
        Y[np.arange(n), y_idx] = 1.0
        return Y

    def fit(self, X, y, X_val=None, y_val=None):
        rng = np.random.default_rng(self.random_state)

        X = sparse.csr_matrix(X)
        n_samples, n_features = X.shape

        self.classes_ = sorted(set(y))
        self.class_to_idx_ = {c: i for i, c in enumerate(self.classes_)}
        n_classes = len(self.classes_)
        y_idx = np.array([self.class_to_idx_[label] for label in y])

        self.W = rng.normal(0, 0.01, size=(n_features, n_classes))
        self.b = np.zeros(n_classes)

        n_batches = max(1, n_samples // self.batch_size)
        lr = self.lr

        for epoch in range(1, self.n_epochs + 1):
            perm = rng.permutation(n_samples)
            X_shuf = X[perm]
            y_shuf = y_idx[perm]

            epoch_loss = 0.0
            for b_idx in range(n_batches):
                start = b_idx * self.batch_size
                end = start + self.batch_size if b_idx < n_batches - 1 else n_samples
                if start >= end:
                    continue
                Xb = X_shuf[start:end]
                yb = y_shuf[start:end]
                m = Xb.shape[0]

                Z = Xb.dot(self.W) + self.b
                P = softmax(Z)
                Yb = self._one_hot(yb, n_classes)

                eps = 1e-12
                loss = -np.sum(Yb * np.log(P + eps)) / m
                loss += (self.l2 / (2 * m)) * np.sum(self.W * self.W)
                epoch_loss += loss * m

                dZ = (P - Yb) / m
                dW = Xb.T.dot(dZ)
                dW = np.asarray(dW) + (self.l2 / m) * self.W
                db = dZ.sum(axis=0)

                self.W -= lr * dW
                self.b -= lr * db

            lr *= self.lr_decay
            if self.verbose and (epoch == 1 or epoch % 5 == 0 or epoch == self.n_epochs):
                msg = f"epoch {epoch:3d}/{self.n_epochs}  train_loss={epoch_loss/n_samples:.4f}"
                if X_val is not None and y_val is not None:
                    val_acc = (self.predict(X_val) == np.asarray(y_val)).mean()
                    msg += f"  val_acc={val_acc:.4f}"
                print(msg)
        return self

    def predict_proba(self, X):
        X = sparse.csr_matrix(X)
        Z = X.dot(self.W) + self.b
        return softmax(Z)

    def predict(self, X):
        P = self.predict_proba(X)
        idx = np.argmax(P, axis=1)
        return np.array([self.classes_[i] for i in idx])


## 5. Custom evaluation metrics (from scratch)

Per-class TP/FP/FN → precision/recall/F1 per class → macro-F1 = unweighted mean. No `sklearn.metrics.*` is used.

In [39]:
def confusion_counts(y_true, y_pred, labels):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    tp, fp, fn = {}, {}, {}
    for lab in labels:
        tp[lab] = int(np.sum((y_true == lab) & (y_pred == lab)))
        fp[lab] = int(np.sum((y_true != lab) & (y_pred == lab)))
        fn[lab] = int(np.sum((y_true == lab) & (y_pred != lab)))
    return tp, fp, fn


def precision_recall_f1_per_class(y_true, y_pred, labels):
    tp, fp, fn = confusion_counts(y_true, y_pred, labels)
    precision, recall, f1 = {}, {}, {}
    for lab in labels:
        p_den = tp[lab] + fp[lab]
        r_den = tp[lab] + fn[lab]
        p = tp[lab] / p_den if p_den > 0 else 0.0
        r = tp[lab] / r_den if r_den > 0 else 0.0
        f = 2 * p * r / (p + r) if (p + r) > 0 else 0.0
        precision[lab], recall[lab], f1[lab] = p, r, f
    return precision, recall, f1


def macro_f1_score(y_true, y_pred, labels=None):
    if labels is None:
        labels = sorted(set(list(y_true) + list(y_pred)))
    _, _, f1 = precision_recall_f1_per_class(y_true, y_pred, labels)
    return float(np.mean([f1[lab] for lab in labels]))


def accuracy_score(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    return float(np.mean(y_true == y_pred))


def classification_report(y_true, y_pred, labels=None):
    if labels is None:
        labels = sorted(set(list(y_true) + list(y_pred)))
    precision, recall, f1 = precision_recall_f1_per_class(y_true, y_pred, labels)
    lines = [f"{'label':20s} {'precision':>10s} {'recall':>10s} {'f1':>10s} {'support':>10s}"]
    y_true_arr = np.asarray(y_true)
    for lab in labels:
        support = int(np.sum(y_true_arr == lab))
        lines.append(f"{str(lab):20s} {precision[lab]:10.4f} {recall[lab]:10.4f} {f1[lab]:10.4f} {support:10d}")
    macro_f1 = np.mean([f1[lab] for lab in labels])
    macro_p = np.mean([precision[lab] for lab in labels])
    macro_r = np.mean([recall[lab] for lab in labels])
    lines.append("")
    lines.append(f"{'macro avg':20s} {macro_p:10.4f} {macro_r:10.4f} {macro_f1:10.4f} {len(y_true):10d}")
    lines.append(f"accuracy: {accuracy_score(y_true, y_pred):.4f}")
    return "\n".join(lines)


## 6. Fit the vectorizer and train the model

In [40]:
print(f"train={len(train_texts)}  val={len(val_texts)}  test={len(test_texts)}")
print(f"num classes: {len(set(train_labels))}")

print("\nFitting TF-IDF vectorizer (word 1-2 grams + char 2/3/4-grams) ...")
vectorizer = CustomTfidfVectorizer(min_df=MIN_DF, max_features=MAX_FEATURES)
X_train = vectorizer.fit_transform(train_texts)
X_val = vectorizer.transform(val_texts)
X_test = vectorizer.transform(test_texts)
print(f"feature matrix shapes: train={X_train.shape} val={X_val.shape} test={X_test.shape}")


train=18400  val=2300  test=2300
num classes: 23

Fitting TF-IDF vectorizer (word 1-2 grams + char 2/3/4-grams) ...
feature matrix shapes: train=(18400, 200000) val=(2300, 200000) test=(2300, 200000)


In [41]:
print("Training custom logistic regression ...")
clf = CustomLogisticRegression(lr=LR, l2=L2, n_epochs=EPOCHS, batch_size=BATCH_SIZE)
clf.fit(X_train, train_labels, X_val=X_val, y_val=val_labels)


Training custom logistic regression ...
epoch   1/30  train_loss=3.0770  val_acc=0.8726
epoch   5/30  train_loss=2.6325  val_acc=0.9383
epoch  10/30  train_loss=2.1913  val_acc=0.9417
epoch  15/30  train_loss=1.8654  val_acc=0.9457
epoch  20/30  train_loss=1.6279  val_acc=0.9452
epoch  25/30  train_loss=1.4532  val_acc=0.9448
epoch  30/30  train_loss=1.3222  val_acc=0.9452


## 7. Evaluate

In [42]:
print("=== Validation performance ===")
val_pred = clf.predict(X_val)
print(f"Macro-F1: {macro_f1_score(val_labels, val_pred):.4f}")
print(f"Accuracy: {accuracy_score(val_labels, val_pred):.4f}")

print("\n=== Test performance ===")
test_pred = clf.predict(X_test)
print(f"Macro-F1: {macro_f1_score(test_labels, test_pred):.4f}")
print(f"Accuracy: {accuracy_score(test_labels, test_pred):.4f}")
print()
print(classification_report(test_labels, test_pred))


=== Validation performance ===
Macro-F1: 0.9408
Accuracy: 0.9452

=== Test performance ===
Macro-F1: 0.9317
Accuracy: 0.9365

label                 precision     recall         f1    support
asm_Beng                 0.9800     0.9800     0.9800        100
ben_Beng                 0.9899     0.9800     0.9849        100
brx_Deva                 0.9875     0.7900     0.8778        100
doi_Deva                 0.9300     0.9300     0.9300        100
gom_Deva                 0.7568     0.2800     0.4088        100
guj_Gujr                 1.0000     1.0000     1.0000        100
hin_Deva                 0.7680     0.9600     0.8533        100
kan_Knda                 1.0000     1.0000     1.0000        100
kas_Arab                 1.0000     0.9700     0.9848        100
khasi                    0.8772     1.0000     0.9346        100
mai_Deva                 0.9438     0.8400     0.8889        100
mal_Mlym                 1.0000     1.0000     1.0000        100
mar_Deva                 0.60